In [2]:
from mp_api.client import MPRester
from datetime import datetime
from pathlib import Path
import json
from monty.json import MontyEncoder, MontyDecoder
from monty.serialization import loadfn, dumpfn


In [3]:
fields_structure = [
    'structure',
 'formula_pretty',
 'volume',
 'density',
 'density_atomic',
 'symmetry',
]
fields_metadata = [
'material_id',
]
fields_electronic = [
 'uncorrected_energy_per_atom',
 'energy_per_atom',
 'energy_above_hull',
 'is_stable',
]
fields = fields_structure + fields_metadata + fields_electronic

In [ ]:
apikey_file = list(Path.home().glob('*API*'))[0]
print(apikey_file)

formula = ['Cu']
with open(apikey_file) as f:
    api_key = f.read().strip()
with MPRester(api_key=api_key) as mpr:
    mpr.session.trust_env = True
    data = mpr.materials.summary.search(formula=formula, 
                                        fields=fields,
                                        is_stable=True
                                       )

# Serialize using MontyEncoder
json_data = json.dumps(data, cls=MontyEncoder)

# Save the JSON string to a file
now = datetime.now().strftime('%Y%m%d-%H%M')
jsonfile = f'materials_data_{now}.json'
with open(jsonfile, 'w') as f:
    f.write(json_data)

/home/jovyan/materials_project_API_KEY


Retrieving SummaryDoc documents:   0%|          | 0/1 [00:00<?, ?it/s]

In [6]:
inputdir = Path('./input')
outputdir = Path('./output')
for _dir in (inputdir, outputdir):
    _dir.mkdir(exist_ok=True)

In [7]:
for d in data:
    s = d.structure
    spacegroup = d.symmetry.symbol.replace('/', 'bar')
    label = '_stable' if d.is_stable else ''
    filename = inputdir/f'{d.material_id}_{s.composition.reduced_formula}_{spacegroup}{label}.cif'
    print(filename)
    s.to(str(filename))
    if d.is_stable:
        s.to(inputdir/'POSCAR')


input/mp-569794_Ta_P4_2barmnm_stable.cif
